# 3.8 数据与实验基础：从 import 到训练循环

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

拆开教程中常见的 recsys_lab.data 与 industrial_experiments：先阅读函数源码，再用真实 MovieLens 行为逐步重写时间切分、张量化、训练、推理和测试，建立从 Notebook 到工程模块的完整调用地图。

## Setup

本 Notebook 的默认真实数据是 **MovieLens latest-small（smoke 确定性切片）；full 用官方完整 MovieLens latest（约 33M 评分）**。`smoke` 档读取仓库内可审计的确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互、曝光、标签或行为序列。切片规则、源地址、哈希与许可记录在 `data/README.md` 及对应数据目录。

**主要资料：** [Python inspect 官方文档](https://docs.python.org/3/library/inspect.html) · [PyTorch Optimizing Model Parameters](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

In [1]:
from pathlib import Path
import os, sys, json
import torch
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_ROOT = Path(os.environ.get("RECSYS_ARTIFACT_ROOT", PROJECT_ROOT)).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("RECSYS_PROFILE", "full")
PROFILE = os.environ["RECSYS_PROFILE"]
CUDA_AVAILABLE = torch.cuda.is_available()
DATASET_KEY = "movielens"
# Setup 只声明执行边界。完整数据由章节 runner 在 Train & Inference 单元按需读取，
# 避免仅打开 Notebook 就解析数千万行文件。
REAL_DATASET = {
    "dataset": DATASET_KEY,
    "profile": PROFILE,
    "loading": "lazy: chapter runner owns loading and returns executed provenance",
    "randomly_fabricated_rows": 0,
}
print({"profile": PROFILE, "project_root": str(PROJECT_ROOT), "artifact_root": str(ARTIFACT_ROOT), "dataset_boundary": REAL_DATASET,
       "cuda_available": CUDA_AVAILABLE,
       "cuda_device": torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

{'profile': 'smoke', 'project_root': '<ARTIFACT_ROOT>', 'artifact_root': '<ARTIFACT_ROOT>', 'dataset_boundary': {'dataset': 'movielens', 'profile': 'smoke', 'loading': 'lazy: chapter runner owns loading and returns executed provenance', 'randomly_fabricated_rows': 0}, 'cuda_available': True, 'cuda_device': 'NVIDIA RTX 6000 Ada Generation'}


## 1. 为什么可以 import，但不能把理解也藏起来

工程代码把反复使用的数据读取、切分、训练和落盘封装成函数，优点是每个算法不用复制几百行相同代码，修复泄漏或指标错误时也只改一处。教程需要再加一层约束：

1. Notebook 正文先解释输入、输出和关键步骤；
2. 用一个小例子逐步重写核心逻辑；
3. 给出函数源码与公式—代码映射；
4. 完整工程文件可在“查看实现源码”或浏览器 IDE 中继续阅读。

调用链不是黑盒：

```text
official CSV -> load_* -> deterministic subset -> time split
             -> task tensors -> model.forward -> loss.backward
             -> inference -> metrics -> results/*.json
```

`import` 只是告诉 Python“函数定义在另一个文件”，并不代表它来自不可见的库。下面让 Python 自己打印函数定义和文件位置。

In [2]:
import inspect
from recsys_lab import data as data_tools
from recsys_lab import industrial_experiments as experiments

for fn in [data_tools.load_movielens, data_tools.leave_last_out, experiments.run_dssm]:
    print(f"\n{fn.__name__}{inspect.signature(fn)}")
    print("defined in:", inspect.getsourcefile(fn))
print("\nleave_last_out source:\n", inspect.getsource(data_tools.leave_last_out))
loader_lines = inspect.getsource(data_tools._load_cached).splitlines()
print("\nMovieLens loader 的前 36 行（完整函数可在源码讲解页查看）：\n", "\n".join(loader_lines[:36]))


load_movielens(max_users: 'int' = 120, max_items: 'int' = 600, min_user_events: 'int' = 12) -> 'tuple[pd.DataFrame, pd.DataFrame]'
defined in: <ARTIFACT_ROOT>/recsys_lab/data.py

leave_last_out(ratings: 'pd.DataFrame') -> 'tuple[pd.DataFrame, pd.DataFrame]'
defined in: <ARTIFACT_ROOT>/recsys_lab/data.py

run_dssm(epochs: 'int' = 24, *, progress: 'ProgressCallback | None' = None) -> 'dict'
defined in: <ARTIFACT_ROOT>/recsys_lab/industrial_experiments.py

leave_last_out source:
 def leave_last_out(ratings: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    ordered = ratings.sort_values(["user_id", "timestamp", "item_id"]).copy()
    test_index = ordered.groupby("user_id").tail(1).index
    test = ordered.loc[test_index].sort_values("user_id").copy()
    train = ordered.drop(test_index).copy()
    return train, test


MovieLens loader 的前 36 行（完整函数可在源码讲解页查看）：
 @lru_cache(maxsize=8)
def _load_cached(max_users: int, max_items: int, min_user_events: int) -> tuple[pd.DataFrame, pd.DataFr

## 2. 数据加载：确定性切片不是随机造数据

`load_movielens()` 做四件事：读官方 CSV；按行为数稳定选择用户与物品；把原始 ID 映射为连续整数；派生时间、类型和标签列。模型需要连续 ID 来查 embedding，但原始 `userId/movieId` 仍保留以便审计。

下面直接查看真实行和来源记录。`randomly_fabricated_rows=0` 是机器可测试的数据承诺；它不表示数据没有抽样，而表示所有保留行都能追溯到官方文件。

In [3]:
from recsys_lab.data import load_movielens, movielens_provenance

ratings, movies = load_movielens(max_users=12, max_items=80, min_user_events=8)
display(ratings[["userId", "movieId", "user_id", "item_id", "rating", "timestamp"]].head())
print(movielens_provenance(ratings))
assert len(ratings) > 0 and ratings.userId.notna().all()

,userId,movieId,user_id,item_id,rating,timestamp
0,414,1210,6,33,5.0,961436248
1,414,2997,6,59,5.0,961436515
2,414,2355,6,49,4.0,961436556
3,414,2716,6,52,5.0,961436763
4,414,3578,6,63,5.0,961436763


{'dataset': 'MovieLens latest-small (GroupLens, generated 2018-09-26)', 'source': 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', 'license_file': '<ARTIFACT_ROOT>/data/ml-latest-small/README.txt', 'profile': 'smoke', 'slice_rule': 'deterministic user/item subset', 'rows_used': 914, 'users_used': 12, 'items_used': 80, 'time_min_utc': '2000-06-19T17:37:28+00:00', 'time_max_utc': '2018-02-20T10:34:44+00:00', 'positive_rule': 'like := observed rating >= 4.0; very_like := observed rating >= 4.5', 'randomly_fabricated_rows': 0}


## 3. 时间切分：先手写，再与工具函数核对

leave-last-out 对每位用户保留时间最晚的一条作为测试目标，其余进入训练。排序键中加入 `item_id`，是为了在时间戳相同时仍得到稳定结果。关键原则是用过去预测未来；如果随机切分，未来行为可能泄漏到用户历史。

公式不复杂。对用户 $u$ 的事件时间集合 $T_u$，测试事件索引为 $t_u^*=\arg\max_{t\in T_u}t$，训练集为其余事件。

In [4]:
# 逐步重写工具函数
ordered = ratings.sort_values(["user_id", "timestamp", "item_id"]).copy()
last_indices = ordered.groupby("user_id").tail(1).index
manual_test = ordered.loc[last_indices].sort_values("user_id")
manual_train = ordered.drop(last_indices)

helper_train, helper_test = data_tools.leave_last_out(ratings)
assert manual_test[["user_id", "item_id"]].reset_index(drop=True).equals(
    helper_test[["user_id", "item_id"]].reset_index(drop=True)
)
print({"train_rows": len(manual_train), "test_rows": len(manual_test),
       "one_test_row_per_user": bool((manual_test.groupby("user_id").size() == 1).all())})

{'train_rows': 902, 'test_rows': 12, 'one_test_row_per_user': True}


## 4. 从 DataFrame 到张量

DataFrame 适合检查列和时间；模型计算需要张量。下面把 user/item ID 变成整数张量，把评分变成浮点目标。若有 $B$ 行样本、embedding 维数为 $d$，查表后 user/item 形状都是 $[B,d]$，逐维相乘再求和得到 $[B]$ 个预测。

In [5]:
import torch

train_users = torch.tensor(manual_train.user_id.to_numpy(), dtype=torch.long)
train_items = torch.tensor(manual_train.item_id.to_numpy(), dtype=torch.long)
train_targets = torch.tensor(manual_train.rating.to_numpy(), dtype=torch.float32)
print({"users": tuple(train_users.shape), "items": tuple(train_items.shape),
       "targets": tuple(train_targets.shape), "dtypes": [str(train_users.dtype), str(train_targets.dtype)]})

{'users': (902,), 'items': (902,), 'targets': (902,), 'dtypes': ['torch.int64', 'torch.float32']}


## 5. 训练循环：forward、loss、backward、step

训练循环的五个动作适用于 DSSM、DeepFM、DIN 和 HSTU，只是 `forward` 与损失定义不同：

1. `model(...)` 计算预测；
2. `loss_fn` 把预测错误压成一个数；
3. `zero_grad()` 清除上一轮梯度；
4. `backward()` 按链式法则计算每个参数对损失的影响；
5. `step()` 沿降低损失的方向更新参数。

这里用最小矩阵分解示范。预测 $\hat r_{ui}=p_u^\top q_i$，均方误差 $L=\frac1B\sum(\hat r-r)^2$。代码使用的每一行都来自真实评分。

In [6]:
class TinyMF(torch.nn.Module):
    def __init__(self, users, items, dim=8):
        super().__init__()
        self.user = torch.nn.Embedding(users, dim)
        self.item = torch.nn.Embedding(items, dim)
    def forward(self, user_id, item_id):
        return (self.user(user_id) * self.item(item_id)).sum(dim=1)

torch.manual_seed(7)
model = TinyMF(int(ratings.user_id.max()) + 1, int(ratings.item_id.max()) + 1)
optimizer = torch.optim.Adam(model.parameters(), lr=.03)
loss_fn = torch.nn.MSELoss()
losses = []
for epoch in range(20):
    prediction = model(train_users, train_items)       # forward
    loss = loss_fn(prediction, train_targets)          # score the error
    optimizer.zero_grad()                              # clear old gradients
    loss.backward()                                    # chain rule
    optimizer.step()                                   # update parameters
    losses.append(float(loss.detach()))
print({"loss_start": round(losses[0], 4), "loss_end": round(losses[-1], 4)})
assert losses[-1] < losses[0]

{'loss_start': 24.2931, 'loss_end': 10.3478}


## 6. 推理与测试为什么必须分开

推理只做前向计算，不修改参数，因此使用 `model.eval()` 和 `torch.no_grad()`。测试集 RMSE 为

$$\mathrm{RMSE}=\sqrt{\frac1N\sum_{n=1}^N(\hat r_n-r_n)^2}.$$

它回答“评分数值差多少”，不能替代 Top-K 召回指标。深度算法 Notebook 也遵循同样边界：训练函数可以封装，但测试目标、候选集和指标公式必须在正文中说清。

In [7]:
test_users = torch.tensor(helper_test.user_id.to_numpy(), dtype=torch.long)
test_items = torch.tensor(helper_test.item_id.to_numpy(), dtype=torch.long)
test_targets = torch.tensor(helper_test.rating.to_numpy(), dtype=torch.float32)
model.eval()
with torch.no_grad():
    test_prediction = model(test_users, test_items)
    rmse = torch.sqrt(torch.mean((test_prediction - test_targets) ** 2)).item()
print({"test_rows": len(test_targets), "RMSE": round(rmse, 4)})
assert torch.isfinite(test_prediction).all()

{'test_rows': 12, 'RMSE': 3.8892}


## 7. 怎样阅读 `run_dssm` 这类完整实验

不要从第一行读到最后一行。按以下顺序定位：

1. `_real_amazon`：数据从哪里来；
2. 时间切分与 `fields`：一行表如何变成张量；
3. `DSSM(...)`：模型结构和超参数；
4. `_train_binary`：损失与反向传播；
5. `model.mode = user/item`：两座塔如何分开推理；
6. `_recall_single_target`：全库分数如何变成 Recall@K。

网页顶部的“查看实现源码”会按这些函数分段显示；“在 IDE 中打开”适合跨文件搜索、跳转定义和临时修改。教程正文解释设计，源码页解释实现，IDE 负责自由探索，三者各司其职。

## Checks

本章已经用真实数据验证加载、时间切分、张量化、训练、推理和测试；也证明手写切分与公共工具一致。

In [8]:
assert REAL_DATASET['randomly_fabricated_rows'] == 0
assert len(helper_test) == ratings.user_id.nunique()
assert losses[-1] < losses[0]
print('PASS：公共管线已逐步展开并完成端到端验证。')

PASS：公共管线已逐步展开并完成端到端验证。


## Next Steps

回到任一深度模型 Notebook，先读 Model Structure & Formula Walkthrough，再用源码讲解页把公式对应到实际函数；需要修改实现时进入浏览器 IDE。